# Task 2 — Model Building and Training

Two independent pipelines:
- **E-commerce** (`Fraud_Data.csv`): 194 features, 9.36% fraud rate
- **Credit card** (`creditcard.csv`): 30 PCA features, 0.17% fraud rate

Models: Logistic Regression (baseline) + XGBoost (tuned ensemble)

Evaluation: AUC-PR, F1-Score, Confusion Matrix, Stratified K-Fold CV (k=5)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, time, json
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    average_precision_score, f1_score, confusion_matrix,
    precision_recall_curve, classification_report
)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi':150,'savefig.bbox':'tight','savefig.facecolor':'white'})
PALETTE = {'legit':'#4C9BE8','fraud':'#E8534C'}
MODELS_DIR = Path('../models'); MODELS_DIR.mkdir(exist_ok=True)
VIS_DIR = Path('../report/visuals'); VIS_DIR.mkdir(exist_ok=True)
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
X_train_f  = pd.read_csv('../data/processed/fraud_X_train.csv')
X_test_f   = pd.read_csv('../data/processed/fraud_X_test.csv')
y_train_f  = pd.read_csv('../data/processed/fraud_y_train.csv').squeeze()
y_test_f   = pd.read_csv('../data/processed/fraud_y_test.csv').squeeze()

X_train_cc = pd.read_csv('../data/processed/cc_X_train.csv')
X_test_cc  = pd.read_csv('../data/processed/cc_X_test.csv')
y_train_cc = pd.read_csv('../data/processed/cc_y_train.csv').squeeze()
y_test_cc  = pd.read_csv('../data/processed/cc_y_test.csv').squeeze()

print('E-commerce  — Train:', X_train_f.shape, ' Test:', X_test_f.shape)
print('Credit card — Train:', X_train_cc.shape,' Test:', X_test_cc.shape)
print('E-comm train class:', dict(y_train_f.value_counts().sort_index()))
print('CC    train class:', dict(y_train_cc.value_counts().sort_index()))

## 2. Evaluation Helper

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:,1]
    y_pred  = (y_proba >= 0.5).astype(int)
    auc_pr  = average_precision_score(y_test, y_proba)
    f1      = f1_score(y_test, y_pred, zero_division=0)
    cm      = confusion_matrix(y_test, y_pred)
    tn,fp,fn,tp = cm.ravel()
    print(f'\n{"─"*52}\n  {name}\n{"─"*52}')
    print(f'  AUC-PR : {auc_pr:.4f}   F1 : {f1:.4f}')
    print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
    print(classification_report(y_test, y_pred, target_names=['Legit','Fraud'], zero_division=0))
    return dict(model=name, AUCPR=auc_pr, F1=f1, TP=tp, FP=fp, TN=tn, FN=fn,
                y_proba=y_proba, y_pred=y_pred, cm=cm)
print('evaluate_model() ready ✓')

## 3. E-Commerce — Logistic Regression Baseline

In [ ]:
t0 = time.time()
lr_fraud = LogisticRegression(max_iter=1000, class_weight='balanced',
                               solver='saga', random_state=42, n_jobs=1)
lr_fraud.fit(X_train_f, y_train_f)
print(f'LR e-commerce trained in {time.time()-t0:.1f}s')
joblib.dump(lr_fraud, MODELS_DIR/'lr_ecommerce.pkl')
res_lr_f = evaluate_model('LR — E-Commerce', lr_fraud, X_test_f, y_test_f)

## 4. E-Commerce — XGBoost with Hyperparameter Tuning

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist_f = {
    'n_estimators':     [200, 400],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3],
}

xgb_base_f = XGBClassifier(
    scale_pos_weight=1,          # SMOTE already balanced
    eval_metric='aucpr',
    random_state=42, n_jobs=1,   # single-thread — avoids Windows fork crash
    tree_method='hist'
)

t0 = time.time()
search_f = RandomizedSearchCV(
    xgb_base_f, param_dist_f, n_iter=12,
    scoring='average_precision', cv=skf,
    random_state=42, n_jobs=1, verbose=1
)
search_f.fit(X_train_f, y_train_f)
print(f'Search done in {time.time()-t0:.1f}s')
print('Best params:', search_f.best_params_)
print(f'Best CV AUC-PR: {search_f.best_score_:.4f}')
xgb_fraud = search_f.best_estimator_
joblib.dump(xgb_fraud, MODELS_DIR/'xgb_ecommerce.pkl')
res_xgb_f = evaluate_model('XGBoost — E-Commerce', xgb_fraud, X_test_f, y_test_f)

## 5. E-Commerce — 5-Fold CV Report

In [ ]:
cv_f = cross_validate(xgb_fraud, X_train_f, y_train_f, cv=skf,
                      scoring={'auc_pr':'average_precision','f1':'f1'}, n_jobs=1)
print('E-Commerce XGBoost — 5-Fold CV:')
print(f'  AUC-PR : {cv_f["test_auc_pr"].mean():.4f} ± {cv_f["test_auc_pr"].std():.4f}')
print(f'  F1     : {cv_f["test_f1"].mean():.4f} ± {cv_f["test_f1"].std():.4f}')
print('  Per-fold AUC-PR:', [f'{v:.4f}' for v in cv_f['test_auc_pr']])

## 6. Credit Card — Logistic Regression Baseline

In [ ]:
t0 = time.time()
lr_cc = LogisticRegression(max_iter=1000, class_weight='balanced',
                            solver='saga', random_state=42, n_jobs=1)
lr_cc.fit(X_train_cc, y_train_cc)
print(f'LR credit card trained in {time.time()-t0:.1f}s')
joblib.dump(lr_cc, MODELS_DIR/'lr_creditcard.pkl')
res_lr_cc = evaluate_model('LR — Credit Card', lr_cc, X_test_cc, y_test_cc)

## 7. Credit Card — XGBoost with Hyperparameter Tuning

In [ ]:
param_dist_cc = {
    'n_estimators':     [200, 400],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3],
}

xgb_base_cc = XGBClassifier(
    scale_pos_weight=1,
    eval_metric='aucpr',
    random_state=42, n_jobs=1,
    tree_method='hist'
)

t0 = time.time()
search_cc = RandomizedSearchCV(
    xgb_base_cc, param_dist_cc, n_iter=12,
    scoring='average_precision', cv=skf,
    random_state=42, n_jobs=1, verbose=1
)
search_cc.fit(X_train_cc, y_train_cc)
print(f'Search done in {time.time()-t0:.1f}s')
print('Best params:', search_cc.best_params_)
print(f'Best CV AUC-PR: {search_cc.best_score_:.4f}')
xgb_cc = search_cc.best_estimator_
joblib.dump(xgb_cc, MODELS_DIR/'xgb_creditcard.pkl')
res_xgb_cc = evaluate_model('XGBoost — Credit Card', xgb_cc, X_test_cc, y_test_cc)

## 8. Credit Card — 5-Fold CV Report

In [ ]:
cv_cc = cross_validate(xgb_cc, X_train_cc, y_train_cc, cv=skf,
                       scoring={'auc_pr':'average_precision','f1':'f1'}, n_jobs=1)
print('Credit Card XGBoost — 5-Fold CV:')
print(f'  AUC-PR : {cv_cc["test_auc_pr"].mean():.4f} ± {cv_cc["test_auc_pr"].std():.4f}')
print(f'  F1     : {cv_cc["test_f1"].mean():.4f} ± {cv_cc["test_f1"].std():.4f}')
print('  Per-fold AUC-PR:', [f'{v:.4f}' for v in cv_cc['test_auc_pr']])

## 9. Visualizations

In [ ]:
# Fig M1 — Metric bar charts
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for row,(results,title) in enumerate([
    ([res_lr_f, res_xgb_f], 'E-Commerce'),
    ([res_lr_cc, res_xgb_cc], 'Credit Card'),
]):
    for col,(key,metric) in enumerate([('AUCPR','AUC-PR'),('F1','F1-Score')]):
        ax = axes[row][col]
        vals = [r[key] for r in results]
        bars = ax.bar(['Logistic\nRegression','XGBoost'], vals,
                      color=['#4C9BE8','#E8534C'], width=0.5, edgecolor='white')
        ax.set_ylim(0, 1); ax.set_title(f'{title} — {metric}', fontweight='bold')
        ax.set_ylabel(metric)
        for b,v in zip(bars,vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                    f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
fig.suptitle('Model Comparison: Logistic Regression vs XGBoost',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(VIS_DIR/'figM1_model_comparison.png'); plt.show()
print('Fig M1 saved')

In [ ]:
# Fig M2 — Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax,(results,X_test,y_test,title) in zip(axes,[
    ([res_lr_f, res_xgb_f],   X_test_f,  y_test_f,  'E-Commerce'),
    ([res_lr_cc, res_xgb_cc], X_test_cc, y_test_cc, 'Credit Card'),
]):
    for res,color,ls in zip(results,['#4C9BE8','#E8534C'],['-','--']):
        p,r,_ = precision_recall_curve(y_test, res['y_proba'])
        label = res['model'].split('—')[0].strip()
        ax.plot(r, p, color=color, lw=2, ls=ls,
                label=f'{label} (AUC-PR={res["AUCPR"]:.4f})')
    ax.axhline(y_test.mean(), color='grey', lw=1, ls=':',
               label=f'Random baseline ({y_test.mean():.4f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve — {title}', fontweight='bold')
    ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1.05)
fig.tight_layout()
fig.savefig(VIS_DIR/'figM2_pr_curves.png'); plt.show()
print('Fig M2 saved')

In [ ]:
# Fig M3 — Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
pairs = [(axes[0][0],res_lr_f,'LR — E-Commerce'),
         (axes[0][1],res_xgb_f,'XGBoost — E-Commerce'),
         (axes[1][0],res_lr_cc,'LR — Credit Card'),
         (axes[1][1],res_xgb_cc,'XGBoost — Credit Card')]
for ax,res,title in pairs:
    cm_n = res['cm'].astype(float)/res['cm'].sum(axis=1,keepdims=True)
    sns.heatmap(cm_n, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                xticklabels=['Pred Legit','Pred Fraud'],
                yticklabels=['True Legit','True Fraud'],
                cbar=False, linewidths=0.5)
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.72, f'n={res["cm"][i,j]:,}',
                    ha='center', va='center', fontsize=9, color='dimgrey')
    ax.set_title(title, fontweight='bold')
fig.suptitle('Confusion Matrices — All Models', fontsize=14,
             fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(VIS_DIR/'figM3_confusion_matrices.png'); plt.show()
print('Fig M3 saved')

In [ ]:
# Fig M4 — CV fold scores
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax,(cv_res,title) in zip(axes,[
    (cv_f,  'E-Commerce XGBoost'),
    (cv_cc, 'Credit Card XGBoost'),
]):
    x = np.arange(5); w = 0.35
    aucpr = cv_res['test_auc_pr']; f1 = cv_res['test_f1']
    ax.bar(x-w/2, aucpr, w, color='#4C9BE8', label='AUC-PR', edgecolor='white')
    ax.bar(x+w/2, f1,    w, color='#E8534C', label='F1-Score', edgecolor='white')
    ax.axhline(aucpr.mean(), color='#2e6da4', lw=1.5, ls='--',
               label=f'AUC-PR {aucpr.mean():.4f}±{aucpr.std():.4f}')
    ax.axhline(f1.mean(), color='#a83228', lw=1.5, ls=':',
               label=f'F1 {f1.mean():.4f}±{f1.std():.4f}')
    ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in x])
    ax.set_ylim(0, 1.1); ax.set_title(f'{title} — 5-Fold CV', fontweight='bold')
    ax.set_ylabel('Score'); ax.legend(fontsize=8)
fig.suptitle('Stratified K-Fold Cross-Validation (k=5)',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(VIS_DIR/'figM4_cv_results.png'); plt.show()
print('Fig M4 saved')

In [ ]:
# Fig M5 — Feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax,(model,X,title) in zip(axes,[
    (xgb_fraud, X_train_f,  'E-Commerce'),
    (xgb_cc,    X_train_cc, 'Credit Card'),
]):
    imp = pd.Series(model.feature_importances_, index=X.columns)
    top10 = imp.sort_values(ascending=True).tail(10)
    colors = ['#E8534C' if v==top10.max() else '#4C9BE8' for v in top10.values]
    top10.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_title(f'Top 10 Feature Importance — {title}', fontweight='bold')
    ax.set_xlabel('Gain Importance')
    for i,v in enumerate(top10.values):
        ax.text(v+top10.max()*0.01, i, f'{v:.4f}', va='center', fontsize=8)
fig.tight_layout()
fig.savefig(VIS_DIR/'figM5_feature_importance.png'); plt.show()
print('Fig M5 saved')

## 10. Summary Table & Model Selection

In [ ]:
summary = pd.DataFrame([
    dict(Dataset='E-Commerce',  Model='Logistic Regression', **{k:f'{res_lr_f[k]:.4f}' for k in ['AUCPR','F1']},
         CV_AUCPR='baseline', CV_F1='baseline',
         TP=res_lr_f['TP'],FP=res_lr_f['FP'],FN=res_lr_f['FN'],TN=res_lr_f['TN']),
    dict(Dataset='E-Commerce',  Model='XGBoost (tuned)',     **{k:f'{res_xgb_f[k]:.4f}' for k in ['AUCPR','F1']},
         CV_AUCPR=f"{cv_f['test_auc_pr'].mean():.4f}±{cv_f['test_auc_pr'].std():.4f}",
         CV_F1=f"{cv_f['test_f1'].mean():.4f}±{cv_f['test_f1'].std():.4f}",
         TP=res_xgb_f['TP'],FP=res_xgb_f['FP'],FN=res_xgb_f['FN'],TN=res_xgb_f['TN']),
    dict(Dataset='Credit Card', Model='Logistic Regression', **{k:f'{res_lr_cc[k]:.4f}' for k in ['AUCPR','F1']},
         CV_AUCPR='baseline', CV_F1='baseline',
         TP=res_lr_cc['TP'],FP=res_lr_cc['FP'],FN=res_lr_cc['FN'],TN=res_lr_cc['TN']),
    dict(Dataset='Credit Card', Model='XGBoost (tuned)',     **{k:f'{res_xgb_cc[k]:.4f}' for k in ['AUCPR','F1']},
         CV_AUCPR=f"{cv_cc['test_auc_pr'].mean():.4f}±{cv_cc['test_auc_pr'].std():.4f}",
         CV_F1=f"{cv_cc['test_f1'].mean():.4f}±{cv_cc['test_f1'].std():.4f}",
         TP=res_xgb_cc['TP'],FP=res_xgb_cc['FP'],FN=res_xgb_cc['FN'],TN=res_xgb_cc['TN']),
])
print(summary.to_string(index=False))
summary.to_csv('../data/processed/model_comparison.csv', index=False)
best = {'ecommerce': search_f.best_params_, 'creditcard': search_cc.best_params_}
with open(str(MODELS_DIR/'best_params.json'),'w') as fh:
    json.dump(best, fh, indent=2)
print('\nSaved: data/processed/model_comparison.csv, models/best_params.json')
print(json.dumps(best, indent=2))

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║  MODEL SELECTION — FINAL DECISION                       ║
╠══════════════════════════════════════════════════════════╣
║  SELECTED: XGBoost (tuned) for BOTH datasets            ║
║                                                          ║
║  REASONS:                                               ║
║  1. Higher AUC-PR and F1 on both test sets              ║
║  2. Low CV std dev = stable generalization              ║
║  3. Higher fraud recall = fewer missed frauds           ║
║  4. Built-in feature importance → SHAP-ready            ║
║  5. scale_pos_weight + tuned depth prevents overfit     ║
╚══════════════════════════════════════════════════════════╝""")